In [1]:
#### INPUT VARIABLE
is_first_run = True

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import sys
import warnings

# Handle ShimWarning import for different IPython versions
if 'IPython' in sys.modules:
    try:
        from IPython.utils.shimmodule import ShimWarning
        warnings.filterwarnings("ignore", category=ShimWarning)
    except ImportError:
        try:
            from IPython.utils.warn import ShimWarning
            warnings.filterwarnings("ignore", category=ShimWarning)
        except ImportError:
            pass


In [4]:
import sys
from pathlib import Path

# Repo root (works on macOS/Linux/Windows when the notebook cwd is the repo)
module_path = str(Path.cwd().resolve())

# Add the module path to sys.path
if module_path not in sys.path:
    sys.path.append(module_path)

from xls_append.main import ArcGISAPI


In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

dotenv_path = Path.cwd().resolve() / '.env'
load_dotenv(dotenv_path=dotenv_path)

True

In [6]:
## initialize the class
arc = ArcGISAPI()

INFO:xls_append.main:ArcGISAPI initialized without XLS_PATH; waiting for runtime DataFrame input
INFO:xls_append.main:ArcGIS API initialized with lazy loading enabled


In [7]:
for layer_group in arc.webmap_data.get('operationalLayers'):
    # print(layer_group)
    for layer in layer_group.get('layers'):
        # print(layer)
        # print(f'title: {layer.get("title")}')
        # print(f'url: {layer.get("url")}')
        print(layer.get('title'))

INFO:xls_append.main:Successfully connected to portal
INFO:xls_append.main:Webmap search successful with 4 webmaps
INFO:xls_append.main:Successfully loaded webmap data


Other countries
Layer AAA100 - Donated Income
Layer BB3 - Conservation Spend
Layer DDD400 - Supporters & Partners
Layer E1 _ Staff
Layer F1 - CO2 Emissions - revision
Layer BBB110 _ Conservation Spend within boundary
Country border - GARA
WWF Country Regional
AAA100 regions all points
BB3 regions all points
BBB110_regions_all_points
DDD400 regions all points
E1_regions_all_points
F1 regions all points
Gara points for cluster


## Clone web map → independent `_stg` / `_dev` (same folder)

Clones stay in the **same portal folder** as the source (root for this project).
Only **titles** are suffixed, e.g.:
- `Catalyze design_GARA_project_data_stg`
- `Catalyze design_GARA_project_data_dev`
- Feature Service: `GAR_stg` / `GAR_dev`

**Before re-running**
1. Interrupt the kernel if a clone is still running
2. Cleanup incomplete Feature Services (dry-run first)
3. Run config → helpers → preview → clone

Progress is written to `./clone_status/clone_*.json`.

In [8]:
# --- clone configuration ---
CLONE_SUFFIXES = ['_stg', '_dev']
SKIP_IF_EXISTS = True

# Keep clones in the source item's folder (None = auto from source; root for this project)
TARGET_FOLDER = None

# progress / failure controls
HEARTBEAT_SEC = 30
STALL_TIMEOUT_SEC = 10 * 60
STATUS_DIR = Path.cwd().resolve() / 'clone_status'

# Fail fast: stop on incomplete data, shared URLs, or leftover weird clones
REQUIRE_CLEAN = True          # abort if incomplete FS clones still exist
EXPECTED_FS_LAYERS = 16       # GAR has 16 layers; set None to skip exact check
STOP_ON_ERROR = True          # do not continue to next suffix after failure

# cleanup cell: True = list only; False = delete incomplete FS clones
CLEANUP_DRY_RUN = True

In [9]:
import logging
from pathlib import Path

from xls_append.clone_webmap import (
    CloneValidationError,
    clone_webmap_environment,
    delete_items,
    extract_item_ids,
    get_webmap_item,
    item_folder_title,
    layer_feature_counts,
    list_incomplete_fs_clones,
    summarize_counts,
    validate_feature_service_rows,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logger = logging.getLogger('clone_webmap')

gis = arc.portal
SOURCE_WEBMAP_TITLE = os.getenv('QUERY')
STATUS_DIR.mkdir(parents=True, exist_ok=True)
print('STATUS_DIR:', STATUS_DIR)
print('SOURCE_WEBMAP_TITLE:', SOURCE_WEBMAP_TITLE)

STATUS_DIR: /Users/treeo/Documents/Github/arcpy_arcgis_api_lib/clone_status
SOURCE_WEBMAP_TITLE: Catalyze design_GARA_project_data


In [10]:
source_webmap = get_webmap_item(gis, SOURCE_WEBMAP_TITLE)
source_folder = TARGET_FOLDER if TARGET_FOLDER is not None else item_folder_title(gis, source_webmap)
related_ids = sorted(extract_item_ids(source_webmap.get_data()))

print('Source web map:', source_webmap.title, source_webmap.id)
print('Target folder:', source_folder or '(root)')
print('Related itemIds:', related_ids)

source_fs_list = []
for iid in related_ids:
    it = gis.content.get(iid)
    print(' ', it.title, '|', it.type, '|', it.id)
    if it.type == 'Feature Service':
        source_fs_list.append(it)
        rows = layer_feature_counts(it)
        summary = validate_feature_service_rows(
            f'source FS {it.title!r}',
            rows,
            require_non_empty=True,
            min_layers=EXPECTED_FS_LAYERS or 1,
        )
        if EXPECTED_FS_LAYERS is not None and summary['total'] != EXPECTED_FS_LAYERS:
            raise CloneValidationError(
                f"source FS has {summary['total']} layers, expected {EXPECTED_FS_LAYERS}"
            )
        print(f"    layers filled {summary['filled']}/{summary['total']} OK")

if not source_fs_list:
    raise CloneValidationError('No Feature Service on source web map — cannot deep-clone data')
if len(source_fs_list) != 1:
    raise CloneValidationError(f'Expected 1 Feature Service, found {len(source_fs_list)}')
source_fs = source_fs_list[0]

print('\nWill create titles:')
for suffix in CLONE_SUFFIXES:
    print(f'  webmap: {source_webmap.title}{suffix}')
    print(f'  FS:     {source_fs.title}{suffix}')

Source web map: Catalyze design_GARA_project_data dee7c09685234ea1805290b2e78d5c5a
Target folder: (root)
Related itemIds: ['6a0cd182e6cc4e67879b381ff01bac57']
  GAR | Feature Service | 6a0cd182e6cc4e67879b381ff01bac57
    layers filled 16/16 OK

Will create titles:
  webmap: Catalyze design_GARA_project_data_stg
  FS:     GAR_stg
  webmap: Catalyze design_GARA_project_data_dev
  FS:     GAR_dev


### Cleanup incomplete Feature Services (same folder)

Deletes incomplete clones whose title is `GAR` / `GAR_stg` / `GAR_dev` (etc.) when layer counts do not match the source. Does **not** create or use `GARA_clones_*` folders.

In [11]:
incomplete = list_incomplete_fs_clones(
    gis,
    source_fs,
    folder=source_folder,
    title_suffixes=CLONE_SUFFIXES,
)
print(f'Incomplete FS clones in {source_folder or "(root)"}: {len(incomplete)}')
for it, summary in incomplete:
    print(f"  - {it.title!r} | {it.id} | filled {summary['filled']}/{summary['total']} empty={summary['empty_layer_names']}")

cleanup_report = delete_items([it for it, _ in incomplete], dry_run=CLEANUP_DRY_RUN)
print('CLEANUP_DRY_RUN =', CLEANUP_DRY_RUN)
cleanup_report

INFO:arcgis.gis._impl._portalpy:getting user folders and items


Incomplete FS clones in (root): 0
CLEANUP_DRY_RUN = True


[]

### Run two-phase clone

Creates `{QUERY}_stg` and `{QUERY}_dev` in the **same folder**, with independent Feature Service data.

In [12]:
clone_results = {}
clone_error = None

for suffix in CLONE_SUFFIXES:
    print('=' * 80)
    print('START', suffix, '->', f'{source_webmap.title}{suffix}')
    try:
        result = clone_webmap_environment(
            gis=gis,
            source_webmap=source_webmap,
            suffix=suffix,
            folder=source_folder,
            status_dir=STATUS_DIR,
            heartbeat_sec=HEARTBEAT_SEC,
            stall_timeout_sec=STALL_TIMEOUT_SEC,
            skip_if_webmap_exists=SKIP_IF_EXISTS,
            require_clean=REQUIRE_CLEAN,
            expected_fs_layers=EXPECTED_FS_LAYERS,
        )
        clone_results[suffix] = result
        if result.get('skipped'):
            print('SKIPPED existing web map', result['webmap'].id)
            continue
        report = result['report']
        print(f"OK {suffix}: webmap={result['webmap'].title} ({result['webmap'].id})")
        print(f"  folder={result.get('folder')}")
        print(f"  feature service={result['feature_service'].title} ({result['feature_service'].id})")
        print(f"  independent_data={report['independent_data']}")
        print(f"  status_file={result['status_file']}")
    except Exception as exc:
        clone_error = exc
        logger.exception('FAILED %s', suffix)
        clone_results[suffix] = {'suffix': suffix, 'error': str(exc)}
        print(f'FAILED {suffix}: {exc}')
        print(f'See status JSON under {STATUS_DIR}')
        if STOP_ON_ERROR:
            print('STOP_ON_ERROR=True — not starting remaining suffixes')
            break

if clone_error and STOP_ON_ERROR:
    raise clone_error

clone_results

START _stg -> Catalyze design_GARA_project_data_stg


INFO:arcgis.gis._impl._portalpy:getting user folders and items
INFO:clone_webmap:status -> /Users/treeo/Documents/Github/arcpy_arcgis_api_lib/clone_status/clone_stg.json
INFO:clone_webmap:status -> /Users/treeo/Documents/Github/arcpy_arcgis_api_lib/clone_status/clone_stg.json
INFO:clone_webmap:status -> /Users/treeo/Documents/Github/arcpy_arcgis_api_lib/clone_status/clone_stg.json
INFO:root:Processing Level: 0
INFO:arcgis.gis._impl._portalpy:getting user folders and items
INFO:clone_webmap:FS clone progress 5f054db14561490faa4acaaa5f706853: filled 0/0 empty=[]
INFO:clone_webmap:status -> /Users/treeo/Documents/Github/arcpy_arcgis_api_lib/clone_status/clone_stg.json
INFO:arcgis.gis._impl._portalpy:getting user folders and items
INFO:clone_webmap:FS clone progress 5f054db14561490faa4acaaa5f706853: filled 0/16 empty=['Gara_points_for_cluster', 'F1 regions all points', 'E1_regions_all_points', 'DDD400_regions_all_points', 'BBB110_regions_all_points', 'BB3 regions all points', 'AAA100 regio

KeyboardInterrupt: 

In [ ]:
for suffix, result in clone_results.items():
    if result.get('error'):
        print(suffix, 'ERROR:', result['error'])
        continue
    wm = result['webmap']
    print(suffix, wm.title, wm.homepage)
    status_file = Path(result.get('status_file', STATUS_DIR / f'clone{suffix}.json'))
    if status_file.exists():
        print(' status:', status_file)
